# TRISTAR Auto Loader Ingestion

This notebook ingests TRISTAR JSON files from Gdańsk using Auto Loader and writes them to a Bronze Delta table.

## Load configuration

Load settings from the YAML configuration file.

In [0]:
import yaml

with open("config.yml", "r") as file:
    config = yaml.safe_load(file)

## Read configuration

Get the catalog, source schema, and volume settings.

In [0]:
catalog = config["catalog"]
source_schema = config["schema"]
volume = config["volume"]


## Define paths

Define the staging, schema, checkpoint paths, and Bronze table.

In [0]:
bronze_schema = f"{source_schema}_bronze"
source_path = f"/Volumes/{catalog}/{source_schema}/{volume}"
dataset_path = f"{source_path}/tristar"

staging_path = f"{dataset_path}/staging"
schema_path = f"{dataset_path}/schema"
checkpoint_path = f"{dataset_path}/checkpoint"

bronze_table = f"{catalog}.{bronze_schema}.tristar"

## Configure Auto Loader

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("rescuedDataColumn", "_rescued_data")
        .option("pathGlobFilter", "*.json")
        .option("multiLine", "true")
        .load(staging_path)
        .withColumn("source_filename", col("_metadata.file_name"))
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("load_date", current_date())
)

## Write to Bronze

Write the stream to the Bronze Delta table using checkpointing and the `availableNow` trigger.

In [0]:

query = (
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()